In [1]:
import os

# 导入markdown拆分对象
from langchain_text_splitters import MarkdownHeaderTextSplitter

# 读取文本中的内容
with open('resources/评估.md','r',encoding='utf-8') as file:
    # 打开文本
    file_content = file.read()

# 创建读写对象
markdown_splitter = MarkdownHeaderTextSplitter(
    # 第一个参数传入切分依据
    [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ],
    # 第二个参数设置是否需要携带标题
    strip_headers=False
)

# 通过对象切分文本,拿到chunks列表,里面是切分好的对象
chunks = markdown_splitter.split_text(file_content)

# 手动设置每个文档片段的id
for index,item  in enumerate(chunks):
    item.id = f'自定义片段id:{index+1}'


In [2]:
from pydantic import SecretStr
from dotenv import load_dotenv
# 创建向量化大模型
from langchain_openai import OpenAIEmbeddings
# 使用基于内存的向量数据库
from langchain_core.vectorstores import InMemoryVectorStore

# 将api_key加载进环境变量
load_dotenv()

api_key = os.getenv('ALI_API_KEY','')
base_url = os.getenv('ALI_BASE_URL')

# 向量模型
ali_embeddings = OpenAIEmbeddings(
    base_url=base_url,
    api_key=SecretStr(api_key),
    model='qwen3.7-text-embedding',
    check_embedding_ctx_length=False  # 取消向量长度检查
)


# 创建向量数据库,指定向量模型
vector_store = InMemoryVectorStore(embedding=ali_embeddings)

In [3]:
# 这里分批是因为向量数据库不接受一次性塞太多数据，单次最多 20 条
# 列表推导式的 range(0, len, 20) 每轮跳 20，把 chunks 切成若干个小列表
# 最后一批如果不够 20 条，切片会自动取剩余的全部，不会报错
document_chunks = [
    chunks[i:i+20]
    for i in range(0, len(chunks), 20)
]

# 批次存储到向量数据库
for doc_chunk in  document_chunks:
    ids = vector_store.add_documents(doc_chunk)
    # 每个文档片段会分配一个id,这个id如果不手动指定,默认用uuid4
    print(ids)

['自定义片段id:1', '自定义片段id:2', '自定义片段id:3', '自定义片段id:4', '自定义片段id:5', '自定义片段id:6', '自定义片段id:7', '自定义片段id:8', '自定义片段id:9', '自定义片段id:10', '自定义片段id:11', '自定义片段id:12', '自定义片段id:13', '自定义片段id:14', '自定义片段id:15', '自定义片段id:16', '自定义片段id:17', '自定义片段id:18', '自定义片段id:19', '自定义片段id:20']
['自定义片段id:21']


In [5]:
# 从向量数据库中检索语义相似度进行提取数据

result_list = vector_store.similarity_search(
    # 向量库通过向量模型对用户提问先进行向量化,然后做语义相似度匹配
    query='我想吃你家大米',
    # 返回五条
    k=5
)

# 打印结果
for doc in result_list:
    # 转json,这个doc是pydantic模型
    print(doc.model_dump_json(indent=2))

{
  "id": "自定义片段id:1",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（一）教育学萌芽时期代表作"
  },
  "page_content": "# 第一章 教育概述  \n## 第一节 中外教育家及其教育思想  \n### （一）教育学萌芽时期代表作  \n国内：《学记》，世界最早，成文于战国末期，作者是乐正克。  \n国外：昆体良《雄辩术原理》/《论演说家的教育》。",
  "type": "Document"
}
{
  "id": "自定义片段id:4",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（四）孟子主要思想"
  },
  "page_content": "### （四）孟子主要思想  \n**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  \n**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  \n**教学原则**：循序渐进，专心有恒。",
  "type": "Document"
}
{
  "id": "自定义片段id:2",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（二）《学记》主要思想"
  },
  "page_content": "### （二）《学记》主要思想  \n**教育作用**：化民成俗，其必由学；建国君民，教学为先。  \n**教学相长**：教和学两方面互相影响和促进，都得到提高。  \n**豫时孙摩**：包括预防性原则、及时性原则、循序渐进原则、集体教育原则。  \n**长善救失**：“学者有四失，教者必知之。人之学也，或失则多，或失则寡，或失则易，或失则止。此四者，心之莫同也。知其心，然后能救其失也，教也者，长善而救其失者也。”  \n**启发诱导**：道而弗牵，强而弗抑，开而弗达。",
  "type": "Document"

In [6]:
# 还可以拿到语义相似度的评分

result_list = vector_store.similarity_search_with_score(
    query='教育概述是什么',
    k = 5,
)
for doc, score in result_list:
    print(f"==========score: {score}=============")
    print(doc.model_dump_json(indent=2))

==========score: 0.6176078283763863=============
{
  "id": "自定义片段id:9",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第二节 教育的定义",
    "Header 3": "（一）教育的定义"
  },
  "page_content": "## 第二节 教育的定义  \n### （一）教育的定义  \n广义的教育是指一切有目的地增进人的知识和技能，发展人的智力和体力，影响人的思想品德的社会活动，具有目的性和社会性。广义教育包括社会教育、家庭教育、学校教育。广义的教育是人类社会有史以来就有的教育活动。  \n狭义的教育就是指学校教育。  \n教育的要素：教育者、受教育者、教育影响（主要是教育内容）。",
  "type": "Document"
}
==========score: 0.5331490636624374=============
{
  "id": "自定义片段id:1",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第一节 中外教育家及其教育思想",
    "Header 3": "（一）教育学萌芽时期代表作"
  },
  "page_content": "# 第一章 教育概述  \n## 第一节 中外教育家及其教育思想  \n### （一）教育学萌芽时期代表作  \n国内：《学记》，世界最早，成文于战国末期，作者是乐正克。  \n国外：昆体良《雄辩术原理》/《论演说家的教育》。",
  "type": "Document"
}
==========score: 0.530337556121882=============
{
  "id": "自定义片段id:10",
  "metadata": {
    "Header 1": "第一章 教育概述",
    "Header 2": "第二节 教育的定义",
    "Header 3": "（二）教育的属性"
  },
  "page_content": "### （二）教育的属性  \n教育的本质属性是有目的地培养人的社会活动。  \n教育的永恒性。  \n教育